# Teach the GPT English first, then The Verdict, then questions

Runs the whole plan on a Colab GPU using the code in your GitHub repository.
No pretrained model is used: the 124M model starts from random weights.

| Stage | What | Rough time on a T4 |
|---|---|---|
| 1 | English: Wikipedia, 131M tokens, 1 epoch | 2-3 hours |
| 2 | The Verdict + 220 fact statements | ~5 min |
| 3 | Question-answer fine-tuning (closed and open book) | ~15 min |
| 4 | Diagnostics | ~5 min |

**Before you start**
1. *Runtime > Change runtime type > T4 GPU*.
2. Only if your GitHub repository is private: add a GitHub token as a Colab secret (click the **key icon** in the left sidebar, add a secret named `GH_TOKEN`, paste the token, and switch on *Notebook access*). A public repository needs no token.
3. You need about 7 GB free in Google Drive. Checkpoints and logs are saved to `MyDrive/gpt-from-scratch/`, so nothing is lost if Colab disconnects.

Run the cells in order. **If Colab disconnects during stage 1, run cells 1-4 again, then the *Resume stage 1* cell** (not the stage 1 cell, which would start over).

In [ ]:
# Cell 1: check the GPU
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0), '| bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
# Cell 2: Google Drive (where checkpoints, token files and logs are kept)
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/gpt-from-scratch'
for sub in ('checkpoints', 'processed', 'logs'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print('Saving to', DRIVE)

In [ ]:
# Cell 3: get the code from GitHub and point its output folders at Drive
import os, subprocess
from google.colab import userdata
REPO = '/content/gpt-from-scratch'
URL = 'github.com/sriya-2005/gpt-from-scratch.git'
if not os.path.exists(REPO):
    try:
        token = userdata.get('GH_TOKEN')  # only needed if the repository is private
    except Exception:
        token = None
    auth = f'x-access-token:{token}@' if token else ''
    subprocess.run(['git', 'clone', '-q', f'https://{auth}{URL}', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)  # forget the token

# models/checkpoints, data/processed and logs live on Drive; copy the repo's small files there once
for local, remote in (('models/checkpoints', 'checkpoints'), ('data/processed', 'processed'), ('logs', 'logs')):
    path = f'{REPO}/{local}'
    if not os.path.islink(path):
        subprocess.run(f'cp -rn {path}/. {DRIVE}/{remote}/ && rm -rf {path} && ln -s {DRIVE}/{remote} {path}',
                       shell=True, check=True)
%cd {REPO}
!git log --oneline -1

In [ ]:
# Cell 4: libraries (Colab already has PyTorch with CUDA; do not install requirements.txt, which pins the CPU build)
!pip install -q transformers tokenizers datasets huggingface_hub tqdm matplotlib pyarrow
!python scripts/check_env.py

In [ ]:
# Cell 5: data. Wikipedia takes ~5-10 min the first time and is then kept on Drive.
import os
if not os.path.exists('data/processed/wikipedia/train.bin'):
    !python scripts/prepare_data.py --dataset wikipedia
!python scripts/prepare_data.py --dataset verdict_full
!python scripts/prepare_data.py --dataset verdict_facts
!python scripts/build_qa_dataset.py

## Stage 1: English

One pass over 131M Wikipedia tokens. Expect the validation loss to start near 10.9 and fall to roughly 4 or below.
Checkpoints are saved every 500 steps. `--time-limit 3.5` stops cleanly (with a checkpoint) before a typical free session ends.

In [ ]:
# Stage 1: start English pretraining from random weights
!python scripts/train.py --preset english --time-limit 3.5

In [ ]:
# Resume stage 1: only if it stopped before reaching the end (disconnect or time limit)
!python scripts/train.py --preset english --resume --time-limit 3.5

In [ ]:
# Look at stage 1: loss curve, and a sample of the English it learned
from IPython.display import Image, display
display(Image('models/checkpoints/english/loss_curve.png'))
!python scripts/inference.py --ckpt models/checkpoints/english/best.pt --prompt "The history of the city" --temperature 0.8 --max-new-tokens 60 --seed 1

## Stage 2: The Verdict + fact statements

Starts from the English model. A lower learning rate (1e-4) than before, so the story is learned without wiping out the English.

In [ ]:
!python scripts/train.py --preset verdict_facts --init-from models/checkpoints/english/best.pt --run-name english_facts --epochs 40 --lr 1e-4
!python scripts/eval_story.py --ckpt models/checkpoints/english_facts/best.pt --trained-frac 1.0

## Stage 3: question-answer fine-tuning

**3a, closed book:** the model answers from memory (`Question: ... Answer:`), kept for comparison.

In [ ]:
!python scripts/finetune_qa.py --init-from models/checkpoints/english_facts/best.pt --rehearsal-data verdict_facts --run-name english_qa

**3b, open book:** each question comes with the story passage the search finds (`Context: ... Question: ... Answer:`), and about 20% with a passage that lacks the answer, so the model learns to say "The story does not say." This is the mode the chat and the desktop app use.

In [ ]:
!python scripts/eval_retrieval.py
!python scripts/finetune_qa.py --with-context --init-from models/checkpoints/english_facts/best.pt --rehearsal-data verdict_facts --run-name english_rag

## Stage 4: diagnostics

Same tests as on the laptop models, for comparison. Laptop round 2 scored: training questions F1 0.75, reworded probe 0.09, validation 0.09; sentence completion (b) about 1/7.

The **test** split is deliberately not used here; it is kept for one final evaluation.

Laptop open book (story-only model, no English stage): validation F1 about 0.09.

In [ ]:
for split in ('train', 'probe', 'val'):
    !python scripts/eval_qa.py --ckpt models/checkpoints/english_qa/best.pt --split {split} --quiet
    !python scripts/eval_qa.py --ckpt models/checkpoints/english_rag/best.pt --split {split} --quiet
!python scripts/eval_cloze.py --ckpt models/checkpoints/english_facts/best.pt models/checkpoints/english_qa/best.pt

## Bring the model to your laptop

In Google Drive, open `My Drive/gpt-from-scratch/checkpoints/english_rag/` (open book) or `english_qa/` (closed book) and download **best.pt** (~470 MB).
On the laptop, put it in `models\checkpoints\english_rag\best.pt` (or `english_qa`), then:

```
.\.venv\Scripts\python.exe scripts\eval_qa.py --ckpt models\checkpoints\english_rag\best.pt --split val
.\.venv\Scripts\python.exe scripts\gui.py models\checkpoints\english_rag\best.pt --ask
```

The logs and CSV files with every answer are in `My Drive/gpt-from-scratch/logs/`.